# Chapter 7 — Key Learnings

This notebook contains my main takeaways, definitions, and conceptual notes
from **Chapter 7 - Fine-tuning to follow instructions** of *'Build a Large Language Model (From Scratch)'* book by Sebastian Raschka.

### 0. Chapter Objective
Fine-tune a pretrained GPT model on instruction–response examples so that it learns to follow natural-language instructions.

### 1. Adding our repo root 'build-llm-from-scratch-pytorch' to sys.path

In [1]:
from pathlib import Path
import sys

# Current folder:
# repository_root/chapter_04/exercises
# i.e. 
# import os  
# print(os.getcwd()) # prints: c:\Users\delmi\Documents\LEARNING\Manning_Learning\repos\build-llm-from-scratch-pytorch\chapter_04\exercises 
# Note: Python searches for chapter_03 (and all other needed imports) inside that folder and in the other locations listed in sys.path. but sys.path currently does not have the repo root
# the snippet below adds the repo root to sys.path

repo_root = Path.cwd().parents[1]

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repository root:", repo_root)
sys.path

Repository root: c:\Users\delmi\Documents\LEARNING\Manning_Learning\repos\build-llm-from-scratch-pytorch


['c:\\Users\\delmi\\Documents\\LEARNING\\Manning_Learning\\repos\\build-llm-from-scratch-pytorch',
 'C:\\Users\\delmi\\anaconda3\\python312.zip',
 'C:\\Users\\delmi\\anaconda3\\DLLs',
 'C:\\Users\\delmi\\anaconda3\\Lib',
 'C:\\Users\\delmi\\anaconda3',
 'c:\\Users\\delmi\\venvs\\llmbookvenv',
 '',
 'c:\\Users\\delmi\\venvs\\llmbookvenv\\Lib\\site-packages']

### 2. format_input() function

In [2]:
def format_input(entry): # Alpaca prompt style instruction + input - entry is a dictionary
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry['input'] else ""
    )
    return instruction_text + input_text

# model_input = format_input(data[50])
# desired_response = f"\n\n### Response:\n{data[50]['output']}"
# print(model_input + desired_response)

**Purpose:** Converts structured `instruction` + optional `input` fields into the prompt format used during instruction fine-tuning.

### 3. InstructionDataset class

In [ ]:
import torch 
from torch.utils.data import Dataset

class InstructionDataset(Dataset): # format Alpaca prompt-style for Instruction + Input + Response for training
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

### InstructionDataset Mental Model

```text
JSON entry
→ format instruction + input + response
→ tokenize complete example
→ store token IDs
```

Each training example teaches:  

instruction + optional input → desired response


The chapter's `InstructionDataset` applies the prompt template (ex: Alpaca or Phi-3 prompt-style template) and pre-tokenizes the examples before batching.

### 4. custom_collate_fn() function
#### Why a Custom Collate Function?

Instruction examples have different lengths, so a batch must:

1. pad examples to a common length;
2. create inputs and next-token targets;
3. replace padding targets with `-100` so they are ignored by cross-entropy loss;
4. optionally truncate sequences to the model's maximum context length.

**batching pipeline:** prompt formatting → tokenization → padding → target creation → replacing padding targets with -100 so they do not contribute to the loss.

```text
Sequence: [A, B, C, EOS, PAD]

Input:    [A, B, C, EOS]
Target:   [B, C, EOS, -100]
```

In [3]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]


        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

### 5. Instruction Fine-Tuning core optimization loop

The core optimization loop is the same next-token training loop implemented in Chapter 5.

What changes is the **training data**:

```text
Chapter 5: Pretraining GPTModel  
raw text → next-token prediction

Chapter 7: Instruction fine-tuning  
instruction + input + target response → next-token prediction
```

>The same cross-entropy objective now teaches the model to generate the desired response given an instruction.

### 6. fine-tuning code
```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=0.1
)

train_losses, val_losses, tokens_seen = train_model_simple(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs=2,
    eval_freq=5,
    eval_iter=5,
    start_context=format_input(val_data[0]),
    tokenizer=tokenizer
)
```

**Key idea:** Instruction fine-tuning updates the pretrained model using
formatted instruction–response pairs rather than raw unlabeled text.

### 7. Response generation

```python
input_text = format_input(entry)

token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer).to(device),
    max_new_tokens=256,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256
)

generated_text = token_ids_to_text(token_ids, tokenizer)

response_text = generated_text[len(input_text):].strip()
```

**Purpose:** Generate from the formatted instruction and remove the original prompt from the decoded text to isolate the model's response.

The chapter later applies this pattern across the test set and stores the extracted model responses for evaluation.

### 8. Chapter 7 Flow

```text
Instruction dataset (JSON)
        ↓
format_input()
        ↓
InstructionDataset
        ↓
custom_collate_fn()
        ↓
[input IDs, target IDs]
        ↓
Pretrained GPT
        ↓
Next-token cross-entropy
        ↓
Instruction fine-tuning
        ↓
generate()
        ↓
Model response
        ↓
Qualitative / automated evaluation
```

### 9. Key definitions (glossary update)

- **Instruction fine-tuning:** Supervised fine-tuning that teaches a pretrained LLM to produce appropriate responses to natural-language instructions.

- **Instruction dataset:** A dataset containing instructions, optional inputs, and expected responses used for supervised fine-tuning.

- **Prompt template:** A consistent textual structure used to format instructions, inputs, and responses for the model.

- **Padding token:** A token added to shorter sequences so examples in the same batch have equal length.

- **Ignore index (`-100`):** A target value ignored by PyTorch cross-entropy loss, used here to prevent padding positions from contributing to training loss.

- **Supervised fine-tuning (SFT):** Additional training of a pretrained model using labeled input–output examples.

### 10. Q/As

- **Q: What is the main difference between pretraining and instruction fine-tuning?**  
  Pretraining learns from raw text, while instruction fine-tuning learns from structured instruction–response examples.

- **Q: Why does instruction fine-tuning require a custom collate function?**  
  Instruction examples have different lengths and must be padded, shifted into input–target pairs, and masked correctly for loss computation.

- **Q: Why are padding targets replaced with `-100`?**  
  PyTorch cross-entropy ignores targets with value `-100`, preventing padding tokens from contributing to the loss.

- **Q: Does instruction fine-tuning require a different GPT architecture?**  
  No. The pretrained GPT architecture remains the same; the model is further trained on instruction–response examples.

- **Q: Why remove the input prompt from generated text during evaluation?**  
  To isolate and evaluate only the model-generated response.